# Custom Summary Network Registration

This notebook defines a tiny custom summary search space, registers it, and runs a minimal one-trial HPO study on a Gaussian simulator.

In [ ]:
from dataclasses import dataclass, field

import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np
from bayesflow_hpo.search_spaces.base import (
    BaseSearchSpace,
    CategoricalDimension,
    IntDimension,
)
from bayesflow_hpo.search_spaces.registry import get_summary_space

In [ ]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

validation_data = hpo.generate_validation_dataset(
    simulator=simulator,
    param_keys=["theta"],
    data_keys=["x"],
    sims_per_condition=16,
)


@dataclass
class TinySummarySpace(BaseSearchSpace):
    summary_dim: IntDimension = field(
        default_factory=lambda: IntDimension("tiny_summary_dim", low=8, high=16, step=8)
    )
    width: IntDimension = field(
        default_factory=lambda: IntDimension(
            "tiny_width", low=32, high=64, step=32, default=False
        )
    )
    pooling: CategoricalDimension = field(
        default_factory=lambda: CategoricalDimension(
            "tiny_pooling", choices=["mean"], default=False
        )
    )

    def build(self, params: dict):
        self._validate(params)
        width = int(params.get("tiny_width", 32))
        return bf.networks.DeepSet(
            summary_dim=int(params["tiny_summary_dim"]),
            depth=2,
            mlp_widths_equivariant=(width, width),
            mlp_widths_invariant_inner=(width, width),
            mlp_widths_invariant_outer=(width, width),
            mlp_widths_invariant_last=(width, width),
            dropout=0.0,
        )


hpo.register_custom_summary_network(
    name="tiny_summary",
    space_factory=TinySummarySpace,
    aliases=["tiny"],
    overwrite=True,
)

hpo.list_registered_network_spaces()

In [ ]:
search_space = hpo.CompositeSearchSpace(
    inference_space=hpo.CouplingFlowSpace(include_optional=False),
    summary_space=get_summary_space("tiny"),
    training_space=hpo.TrainingSpace(include_optional=False),
)

study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["theta"],
    data_keys=["x"],
    validation_data=validation_data,
    search_space=search_space,
    n_trials=1,
    epochs=1,
    batches_per_epoch=1,
    directions=["minimize", "minimize"],
    show_progress_bar=False,
)

print(f"Trials: {len(study.trials)}")
print(f"Best values: {study.best_trials[0].values}")